**Fig. 5, step 1b (Fig. 5A, 5B, Video S5): whole-tissue run on the mouse salivary gland explant with the salivary nucleus model pair**

Needs a GPU.

In [ ]:
# Working directory: data/fig5/ under the repository root, where the Fig. 5 deposit folder is downloaded
# (salivary_gland/ = SG.tif, cell_models/, nuclei_models/ and the engine's epicycle_output_cell/ and
# epicycle_output_nuclei/ trees; human_finetuning/; domain-specialized_finetuned_bootstrap/).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig5")
print("working directory:", os.getcwd())


# EpiCycle ScaledSegmentation Pipeline

Run 3D cell segmentation, reconstruction, and assembly on tiled image data.

**Usage:** Fill in the paths and parameters below, then Run All.

## 1. Paths & Input

In [ ]:
# ===== PATHS (edit these) =====

# Input Z-stack TIFF — set this to tile from scratch, or leave None if already tiled
INPUT_TIF = "salivary_gland/SG.tif"   # Set to None if tiles already exist

# Working directory (root for the project)
WORK_DIR = "salivary_gland"

# CellPose model paths (.pth files)
LATERAL_MODEL = "salivary_gland/nuclei_models/lateral"
AXIAL_MODEL = "salivary_gland/nuclei_models/axial"

# Output directory name (created inside WORK_DIR)
# All outputs go here: tiles/, stage1_segmentation/, stage2_reconstruction/, etc.
OUTPUT_NAME = "epicycle_output_nuclei"

## 2. Resource Configuration

In [ ]:
# ===== RESOURCES (auto-detected from the job's allocation if None) =====

N_GPUS = None            # None = auto-detect from SLURM/CUDA
N_CPU_PER_WORKER = None  # None = auto (total_cores / n_gpus in parallel mode)

## 3. Stage Selection & Tiling Parameters

In [ ]:
# Which stages to run (set False to skip)
RUN_TILING = INPUT_TIF is not None   # Auto: tile if input TIFF provided
RUN_SEGMENTATION = True
RUN_RECONSTRUCTION = True
RUN_ASSEMBLY = True               # coarse: Stage 3 assembly  |  full: the IoU stitch
RUN_GLOBAL_RECONSTRUCTION = True  # coarse mode only (Stage 4 global indirect)

# ===== RECONSTRUCTION MODE =====
#   "coarse" = Mode A: per-tile reconstruct -> stitch -> global INDIRECT recon
#              (factor-2 downsampled, lossy; the original path - byte-identical).
#   "full"   = Mode B: per-tile reconstruct -> full-res IoU SEAM-STITCH
#              (instance-preserving, no downsample). Replaces assembly + global
#              recon; writes stage4_full_stitch/global_labels.tif + .npy +
#              result_manifest.json + label_provenance.csv (the QC worklist).
MODE = "full"             # "coarse" or "full"

# ===== TILING (Stage 0) =====
PATCH_SIZE = 1024           # Target XY patch size in pixels
TILING_OVERLAP = 0.25       # Fractional overlap for tiling
INPUT_AXES = "auto"         # "auto" | "ZCYX" | "CZYX" | "ZYX" - set explicitly for
                            # thin-Z stacks (Z<=C) that lack axes metadata.
Z_THRESHOLD_METHOD = "hard"  # "hard", "percentile", or "default"
Z_THRESHOLD_VALUE = 0.07     # Intensity threshold for Z-range detection
CHANNEL_INDEX = 1            # Which channel for Z-range analysis
AUTO_CORRECT_PATCH = False   # (no-op since flush-anchored tiling - kept for compat)
AUTO_RAM_CAP = True          # shrink the XY tile so ONE tile's recon fits THIS machine's
                             # RAM (no-op on a fat node; prevents OOM on a small box; logs
                             # on shrink -> more seams). False = keep PATCH_SIZE exactly.

# ===== MODE B / FULL - IoU seam-stitch params (used when MODE == "full") =====
IOU_TAU = 0.25              # IoU threshold to match a cell across a seam
IOU_VMIN = 10              # min shared voxels to accept a match (rejects 1-2 vox touches)
IOU_Z_BLOCK = 64           # Z-block for the RAM-bounded chunked matcher
IOU_ONE_TO_MANY = "max_iou"  # "max_iou" (1:1, under-merge bias) or "greedy"
EMIT_PROVENANCE = True     # write label_provenance.csv - per-cell seam/corner QC worklist
                           # (seam_status, min_match_iou rank, ZYX bbox, voxel_count,
                           # touches_border). False = skip it (~1/3 faster write, no QC).

## 3b. Post-Stitch Cleanup + Smooth  *(full mode only)*

In [ ]:
# ===== POST-STITCH CLEANUP + SMOOTH (Stage postprocess; MODE == "full" only) =====
# Runs AFTER the IoU stitch on stage4_full_stitch/global_labels.* + label_provenance.csv.
# Each sub-op is independently toggleable. Writes *_clean outputs next to the stitch
# (non-destructive); idempotent. Section 10b VISUALIZES the size distribution; section 11 RUNS it.
# NOTE: this runs on the EXISTING stitch -- no re-reconstruction / re-stitch needed.
RUN_POSTPROCESS = True

PP_DRY_RUN = False          # True = log drop counts + distribution and write NOTHING
                            #        (calibrate thresholds, then re-run with False)

# Physical voxel size in um as (z, y, x). If set, sizes are reported in um^3 too
# (raw voxel counts are anisotropy-blind; this gives true physical volume).
PP_VOXEL_SIZE_UM = None     # e.g. (1.0, 0.3, 0.3); None = report raw voxels only

# (1) size FLOOR -- drop tiny fragment "confetti".  Give EITHER an absolute voxel
#     count OR a percentile (percentile overrides + auto-adapts -> good when you
#     don't know your sizes yet). Use section 10b to see the distribution + pick.
PP_REMOVE_SMALL = True
PP_MIN_VOXELS = 200         # absolute: drop voxel_count < this
PP_MIN_PERCENTILE = 0.0     # OR percentile: drop the smallest N% (e.g. 2.0); 0 = use absolute

# (1b) size CEILING -- drop extremely large masks (giant blobs / runaway merges).
#      OFF by default (a real big cell and an over-merge look the same by size).
PP_REMOVE_LARGE = False
PP_MAX_VOXELS = 0           # absolute: drop voxel_count > this (0 = off)
PP_MAX_PERCENTILE = 0.0     # OR percentile: drop the largest tail (e.g. 99.5); 0 = use absolute

# (2) detached-island removal -- drop debris not connected to the tissue mass
PP_REMOVE_ISLANDS = True
PP_ISLAND_DOWNSAMPLE = 8    # occupancy-grid stride (bigger = coarser/cheaper)
PP_ISLAND_MIN_FRAC = 0.01   # keep components >= this fraction of the largest (multi-lobe safe)
PP_ISLAND_DILATE = 1        # occupancy dilation iters to bridge close lobes (0 = off)

# (3) per-instance SDF smooth -- mirrors Step-3 curation smooth (topology-preserving)
PP_SMOOTH = True
PP_SMOOTH_SIGMA = 1.5       # isotropic sigma
PP_SMOOTH_SIGMA_Z = None    # override Z sigma (anisotropic s_z>s_xy for Z-staircase); None=isotropic
PP_SMOOTH_SIGMA_XY = None   # override XY sigma; None = isotropic
PP_PRESERVE_TOPOLOGY = True # skeleton-reconnect if a smooth would split a cell

PP_WORKERS = None           # parallel smooth workers (None = allocated CPUs - 1)

## 3c. Per-Cell Feature Extraction  *(full mode only — opt-in)*

In [ ]:
# ===== PER-CELL FEATURE EXTRACTION (Stage features; MODE == "full" only; OPT-IN) =====
# Runs AFTER postprocess on the CLEANED global labels (falls back to the raw stitch).
# Per-cell QC / morphometry -> cell_features.parquet + .csv next to the stitch. Column
# names MIRROR the EpiCycle plugin's qc_features.py (parity). Per-cell bbox-parallel
# (never loads the full volume); idempotent. Section 12 RUNS it.
# NOTE: opt-in + expensive (surface family on ~100k cells) -> dry-run + no-surface first.
RUN_FEATURES = True        # opt-in: set True to extract features after postprocess

FEAT_DRY_RUN = False        # True = log the cell count + size distribution, write NOTHING
FEAT_INCLUDE_SURFACE = True # marching-cubes surface family (the slow part); False = fast pass

# Physical voxel size (z, y, x) um -> area in um^3, lengths/centroids/crowding in um.
FEAT_VOXEL_SIZE_UM = (0.26, 0.26, 0.26)   # e.g. (1.0, 0.3, 0.3); None = voxel units (mirror PP_VOXEL_SIZE_UM)

FEAT_MIN_VOXELS = 1         # skip cells below this (1 = only drop zero-voxel ghosts)
FEAT_INVERT_Z = False       # True if high Z is basal (inverted stacks) - cross-section family
FEAT_K_NEIGHBORS = 6        # K for the centroid mean-KNN crowding distance
FEAT_DENSITY_RADIUS = None  # crowding radius (um if voxel size set); None = auto (3x median NN)
FEAT_PREFER_CLEAN = True    # use *_clean inputs (False = extract off the raw stitch)
FEAT_WORKERS = None         # parallel extract workers (None = allocated CPUs - 1)


## 4. Segmentation Parameters

In [ ]:
# ===== SEGMENTATION (Stage 1) =====

# GPU allocation strategy: "auto", "fixed" (3-GPU triplets), "round_robin"
GPU_STRATEGY = "auto"

# CellPose parameters
FLOW_THRESHOLD = 0.2
CELLPROB_THRESHOLD = 0.0
CHANNELS = [1, 2]      # [signal, nucleus] — adjust for your data )changing for nuclei)
BATCH_SIZE = 1024

## 5. Reconstruction Parameters

In [ ]:
# ===== RECONSTRUCTION (Stage 2) =====

# "sequential" (one tile at a time, all cores) or "parallel" (multiple tiles at
# once, auto-sized from RAM+CPU). Sequential is the safe default; switch to
# "parallel" to actually use a fat node (recon is CPU+RAM bound, ~4 effective
# cores/tile, so one tile leaves most of the node idle).
# RECON_MODE = "sequential"
RECON_MODE = "parallel"

# Max concurrent recon tiles in "parallel" mode. None = AUTO: M = min(cores//4,
# 0.9*RAM/peak_per_tile, n_tiles) -- decoupled from GPU count, since recon barely
# uses the GPU (erosion only). Set an integer to CAP the auto value.
RECON_MAX_WORKERS = None

# ===== PER-TILE GRADIENT-DESCENT (the actual reconstruction integration) =====
# Tune the per-tile u-Segment3D solve (used by Mode B / full). None = tuned defaults
# (n_iter=25, gradient_decay=0.001, momenta=0.98). More iterations / lower decay =
# finer field integration, slower. NOTE: changing these requires RE-RUNNING
# reconstruction (Stage 2) -- they are recon-time params, not post-stitch.
# (Distinct from the Mode-A GLOBAL_N_ITER below, which does NOT apply in full mode.)
RECON_N_ITER = None              # gradient-descent iterations (default 25)
RECON_GRADIENT_DECAY = None      # gradient decay (default 0.001; lower = finer)
RECON_MOMENTA = None             # gradient-descent momentum (default 0.98)

# Post-processing
SAVE_PROB3D = True
SAVE_FLOW3D = False
SAVE_BINARY = True
BINARY_ERODE = True
BINARY_KERNEL_SIZE = 3
BINARY_BORDER_MARGIN = 2
BINARY_FILL_HOLES = True

# Delete stage1 intermediates after reconstruction to save disk
CLEANUP_INTERMEDIATES = False

# ===== ANISOTROPIC PRE-SMOOTHING (optional; default OFF - opt-in) =====
# Smooth per-view flows/probs BEFORE u-Segment3D aggregation to suppress
# slice-to-slice (Z) striations from independent 2D Cellpose predictions. High
# sigma along each view's perpendicular axis, low in-plane. (Byte-identical to
# the EpiCycle monorepo Step 3/4 smoothing.)
PRE_SMOOTH_FLOWS = False         # smooth the per-view gradient fields
PRE_SMOOTH_PROBS = True         # smooth the per-view probability maps
PRE_SMOOTH_SIGMA_PERP = 2.5      # sigma along the slicing-plane-perpendicular axis
PRE_SMOOTH_SIGMA_INPLANE = 1.0   # sigma along the two in-plane axes

## 6. Assembly Parameters  *(coarse mode only)*

In [ ]:
# ===== ASSEMBLY (Stage 3) =====  (coarse mode only)
# These are auto-inferred from tiling_config.json when set to None.
# Only set manually if you tiled outside the pipeline or need overrides.

FULL_SIZE = None    # (H, W) in pixels — auto from tiling
TILE_SIZE = None    # patch size in pixels — auto from tiling
OVERLAP = None      # fractional overlap — auto from tiling
DEPTH = None        # Z slices — auto from tiling

# Which volumes to stitch
STITCH_MASK = True
STITCH_PROB = True
STITCH_BINARY = True

# Overlap blending for probabilities: "average" or "max"
PROB_STITCH_MODE = "max"

# Probability-stitch backend: "numpy" (default) or "zarr" (RAM-bounded chunked —
# avoids the full (depth,H,W) float32 OOM at whole-tissue scale; writes
# full_probabilities.zarr instead of .npy). Nothing downstream reads the
# stitched prob, so the format change is safe.
PROB_BACKEND = "numpy"

## 7. Global Indirect Reconstruction Parameters  *(coarse mode only)*

In [ ]:
# ===== GLOBAL INDIRECT RECONSTRUCTION (Stage 4) =====

DOWNSAMPLE_FACTOR = 2          # Downsample stitched volume by this factor
GLOBAL_MIN_SIZE = 1000         # Remove objects smaller than this (voxels, after downsampling)
USE_PRECOMPUTED_BINARY = True  # Use stitched binary mask as foreground constraint

# Gradient descent parameters
GLOBAL_N_ITER = 150            # Gradient descent iterations (more = finer, slower)
GLOBAL_MOMENTA = 0.98          # Momentum for gradient descent
GLOBAL_GRADIENT_DECAY = 0.1    # Gradient decay (lower = more oversegmentation)
GLOBAL_RENORM_GRADIENT = True  # Renormalize gradient at each step

# Indirect method parameters
GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method
GLOBAL_SMOOTH_SKEL_SIGMA = 1   # Skeleton smoothing sigma
GLOBAL_N_CPU = 64              # CPU threads for multiprocess watershed
GLOBAL_TILE_SHAPE = None       # Tile shape for watershed (None = auto)

## 8. Setup & Pre-flight Check

In [ ]:
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

# Resolve GPU count
from packages.epicycle_scaledseg.utils.resources import get_allocated_gpus
n_gpus = N_GPUS or get_allocated_gpus()

work_dir = Path(WORK_DIR)
work_dir.mkdir(parents=True, exist_ok=True)

# Everything lives inside output_dir: tiles, stage1, stage2, etc.
output_dir = work_dir / OUTPUT_NAME
output_dir.mkdir(exist_ok=True)
log_dir = output_dir / "logs"
log_dir.mkdir(exist_ok=True)

# Tiles directory — inside output_dir alongside stage results
tiles_dir = output_dir / "tiles"

if RUN_TILING:
    print(f"Input TIFF: {INPUT_TIF}")
    print(f"Tiles will be created in: {tiles_dir}")
    print(f"Patch size: {PATCH_SIZE}, overlap: {TILING_OVERLAP}")
else:
    # Discover existing tile directories inside tiles_dir (skip hidden dirs)
    if tiles_dir.is_dir():
        tile_dirs = sorted([
            str(p) for p in tiles_dir.iterdir()
            if p.is_dir() and not p.name.startswith(".")
        ])
    else:
        tile_dirs = []
    print(f"Found {len(tile_dirs)} existing tile directories in {tiles_dir}")
    for td in tile_dirs[:5]:
        print(f"  {Path(td).name}")
    if len(tile_dirs) > 5:
        print(f"  ... and {len(tile_dirs) - 5} more")

print(f"\nGPUs: {n_gpus}")
print(f"Output: {output_dir}")

Input TIFF: <run>/data/SG.tif
Tiles will be created in: <run>/epicycle_output_nuclei/tiles
Patch size: 1024, overlap: 0.25

GPUs: 3
Output: <run>/epicycle_output_nuclei


## 9. Build & Run Pipeline

In [ ]:
from packages.epicycle_scaledseg.core.pipeline import EpiCyclePipeline

pipeline = EpiCyclePipeline(
    work_dir=str(work_dir),
    n_gpus=n_gpus,
    log_dir=str(log_dir),
    mode=MODE,                  # coarse|full; also names the per-mode resume checkpoint
)

# --- Stage 0: Tiling ---
if RUN_TILING:
    from packages.epicycle_scaledseg.core.tiling import TilingStage
    pipeline.add_stage(TilingStage(
        input_tif=INPUT_TIF,
        output_dir=str(tiles_dir),
        patch_size=PATCH_SIZE,
        overlap=TILING_OVERLAP,
        z_threshold_method=Z_THRESHOLD_METHOD,
        threshold_value=Z_THRESHOLD_VALUE,
        channel_index=CHANNEL_INDEX,
        auto_correct=AUTO_CORRECT_PATCH,
        input_axes=INPUT_AXES,
        auto_ram_cap=AUTO_RAM_CAP,
    ))
    # Tiling creates tile_dirs; use placeholder for now
    tile_dirs = [str(work_dir)]

# --- Stage 1: Segmentation ---
if RUN_SEGMENTATION:
    from packages.epicycle_scaledseg.core.segmentation import SegmentationStage
    from packages.epicycle_scaledseg.config.defaults import get_default_segment_params
    # Wire the segmentation (sec. 4) knobs into the params (start from defaults,
    # override). For GRAYSCALE / single-channel data set CHANNELS = [0, 0].
    segment_params = get_default_segment_params()
    segment_params.update({
        "flow_threshold": FLOW_THRESHOLD,
        "cellprob_threshold": CELLPROB_THRESHOLD,
        "channels": CHANNELS,
        "batch_size": BATCH_SIZE,
    })
    pipeline.add_stage(SegmentationStage(
        lateral_model=LATERAL_MODEL,
        axial_model=AXIAL_MODEL,
        segment_params=segment_params,
        out_root=str(output_dir / "stage1_segmentation"),
        gpu_strategy=GPU_STRATEGY,
    ))

# --- Stage 2: Reconstruction ---
if RUN_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.reconstruction import ReconstructionStage
    pipeline.add_stage(ReconstructionStage(
        seg_root=str(output_dir / "stage1_segmentation"),
        out_root=str(output_dir / "stage2_reconstruction"),
        mode=RECON_MODE,
        n_iter=RECON_N_ITER,                  # per-tile gradient-descent overrides
        gradient_decay=RECON_GRADIENT_DECAY,  # (None = tuned defaults 25 / 0.001 / 0.98)
        momenta=RECON_MOMENTA,
        max_workers=RECON_MAX_WORKERS,
        n_cpu_per_worker=N_CPU_PER_WORKER,
        save_prob3d=SAVE_PROB3D,
        save_flow3d=SAVE_FLOW3D,
        save_binary=SAVE_BINARY,
        binary_erode=BINARY_ERODE,
        binary_kernel_size=BINARY_KERNEL_SIZE,
        binary_border_margin=BINARY_BORDER_MARGIN,
        binary_fill_holes=BINARY_FILL_HOLES,
        cleanup_intermediates=CLEANUP_INTERMEDIATES,
        pre_smooth_flows=PRE_SMOOTH_FLOWS,
        pre_smooth_probs=PRE_SMOOTH_PROBS,
        pre_smooth_sigma_perp=PRE_SMOOTH_SIGMA_PERP,
        pre_smooth_sigma_inplane=PRE_SMOOTH_SIGMA_INPLANE,
    ))

# --- Stage 3+: stitch (mode-dependent) ---
if MODE == "coarse":
    # Mode A: Assembly (Stage 3) + Global indirect reconstruction (Stage 4).
    # Assembly params (full_size/tile_size/overlap/depth) auto-infer from
    # tiling_config.json. Pass explicit values above only to override.
    if RUN_ASSEMBLY:
        from packages.epicycle_scaledseg.core.assembly import AssemblyStage
        pipeline.add_stage(AssemblyStage(
            recon_root=str(output_dir / "stage2_reconstruction"),
            output_dir=str(output_dir / "stage3_assembly"),
            full_size=FULL_SIZE,
            tile_size=TILE_SIZE,
            overlap=OVERLAP,
            depth=DEPTH,
            stitch_mask=STITCH_MASK,
            stitch_prob=STITCH_PROB and SAVE_PROB3D,
            stitch_binary=STITCH_BINARY and SAVE_BINARY,
            prob_stitch_mode=PROB_STITCH_MODE,
            prob_backend=PROB_BACKEND,
        ))
    if RUN_GLOBAL_RECONSTRUCTION:
        from packages.epicycle_scaledseg.core.global_reconstruction import GlobalReconstructionStage
        from packages.epicycle_scaledseg.config.defaults import get_default_indirect_reconstruct_params

        indirect_params = get_default_indirect_reconstruct_params(n_cpu=GLOBAL_N_CPU)
        indirect_params['gradient_descent']['n_iter'] = GLOBAL_N_ITER
        indirect_params['gradient_descent']['momenta'] = GLOBAL_MOMENTA
        indirect_params['gradient_descent']['gradient_decay'] = GLOBAL_GRADIENT_DECAY
        indirect_params['gradient_descent']['renorm_gradient'] = GLOBAL_RENORM_GRADIENT
        indirect_params['indirect_method']['dtform_method'] = GLOBAL_DTFORM_METHOD
        indirect_params['indirect_method']['smooth_skel_sigma'] = GLOBAL_SMOOTH_SKEL_SIGMA
        if GLOBAL_TILE_SHAPE is not None:
            indirect_params['gradient_descent']['tile_shape'] = GLOBAL_TILE_SHAPE

        pipeline.add_stage(GlobalReconstructionStage(
            assembly_dir=str(output_dir / "stage3_assembly"),
            output_dir=str(output_dir / "stage4_global"),
            downsample_factor=DOWNSAMPLE_FACTOR,
            min_size=GLOBAL_MIN_SIZE,
            use_precomputed_binary=USE_PRECOMPUTED_BINARY,
            indirect_params=indirect_params,
            n_cpu=GLOBAL_N_CPU,
            tile_shape=GLOBAL_TILE_SHAPE,
        ))

elif MODE == "full":
    # Mode B: full-resolution IoU seam-stitch - replaces assembly + global recon.
    # Reads the per-tile masks + tiling_config tile_geometry, matches cells across
    # seams by IoU, and writes global_labels.tif/.npy + result_manifest.json +
    # (when EMIT_PROVENANCE) label_provenance.csv. The Stage-4 GLOBAL_* params and
    # the Stage-3 PROB/BINARY params do NOT apply here.
    if RUN_ASSEMBLY:
        from packages.epicycle_scaledseg.core.iou_stitch_stage import IoUStitchStage
        pipeline.add_stage(IoUStitchStage(
            recon_root=str(output_dir / "stage2_reconstruction"),
            output_dir=str(output_dir / "stage4_full_stitch"),
            tau=IOU_TAU,
            vmin=IOU_VMIN,
            one_to_many=IOU_ONE_TO_MANY,
            z_block=IOU_Z_BLOCK,
            emit_provenance=EMIT_PROVENANCE,
        ))

else:
    raise ValueError(f"MODE must be 'coarse' or 'full', got {MODE!r}")

print(f"Pipeline ({MODE} mode): {len(pipeline.stages)} stages")
for s in pipeline.stages:
    print(f"  - {s.name}")
print(f"Tile dirs: {len(tile_dirs)}")
print(f"Output: {output_dir}")
if MODE == "full":
    print(f"Mode B final output -> {output_dir / 'stage4_full_stitch' / 'global_labels.tif'}")
else:
    print(f"Mode A final output -> {output_dir / 'stage4_global' / 'global_segmentation.tif'}")

# ===== RUN =====
result = pipeline.run(tile_dirs)
print()
print(result.summary())

2026-06-19 15:03:06,165 epicycle.pipeline INFO Pipeline starting: 4 stage(s), 1 tile(s), 3 GPU(s)
2026-06-19 15:03:06,166 epicycle.pipeline INFO Run parameters saved to <run>/epicycle_output_nuclei/logs/run_params.json
2026-06-19 15:03:06,167 epicycle.pipeline INFO === Stage: tiling ===
2026-06-19 15:03:06,168 epicycle.tiling INFO Stage 'tiling' starting with 1 tile(s)
2026-06-19 15:03:06,168 epicycle.tiling INFO Parameters: {"input_tif": "<run>/data/SG.tif", "patch_size": 1024, "overlap": 0.25, "z_threshold_method": "hard", "threshold_value": 0.07, "channel_index": 1, "auto_correct": false, "input_axes": "auto", "auto_ram_cap": true}
2026-06-19 15:03:06,169 epicycle.core.tiling INFO Tiling already complete (<run>/epicycle_output_nuclei/tiles/tiling_config.json), skipping
2026-06-19 15:03:06,170 epicycle.pipeline INFO Stage 'tiling': 42/42 succeeded (0.0s)
2026-06-19 15:03:06,171 epicycle.tiling INFO Stage 'tiling' finished: 42/42 succeeded (0.0s)
2026-06-19 15:03:06,172 epicycle.pipel

Pipeline (full mode): 4 stages
  - tiling
  - segmentation
  - reconstruction
  - iou_stitch
Tile dirs: 1
Output: <run>/epicycle_output_nuclei
Mode B final output -> <run>/epicycle_output_nuclei/stage4_full_stitch/global_labels.tif


2026-06-19 15:13:22,545 epicycle.iou_stitch INFO Mode-B stitch complete: 87411 global label(s) -> global_labels.npy + global_labels.tif
2026-06-19 15:13:22,585 epicycle.pipeline INFO Stage 'iou_stitch': 42/42 succeeded (616.4s)
2026-06-19 15:13:22,585 epicycle.iou_stitch INFO Stage 'iou_stitch' finished: 42/42 succeeded (616.4s)
2026-06-19 15:13:22,586 epicycle.pipeline INFO Pipeline: 42/42 tiles (616.4s total)
  tiling: 42/42 succeeded (0.0s)
  segmentation: 42/42 succeeded (0.0s)
  reconstruction: 42/42 succeeded (0.0s)
  iou_stitch: 42/42 succeeded (616.4s)



Pipeline: 42/42 tiles (616.4s total)
  tiling: 42/42 succeeded (0.0s)
  segmentation: 42/42 succeeded (0.0s)
  reconstruction: 42/42 succeeded (0.0s)
  iou_stitch: 42/42 succeeded (616.4s)


## 10. Check Status

## 10b. Cell-Size Distribution  *(full mode — calibrate the cleanup thresholds)*

See the size distribution of *all* stitched cells before cleaning. Read off where the confetti / real cells / giant blobs separate, then set `PP_MIN_*` / `PP_MAX_*` (absolute or percentile) in section 3b. Provide `PP_VOXEL_SIZE_UM` to read sizes in µm³.

In [ ]:
# ===== Cell-size distribution (calibrate the postprocess thresholds) =====
# Reads the RAW stitch provenance so you see ALL cells (incl. junk) BEFORE cleaning.
# Eyeball the gaps: confetti (far left) | real cells (the bulk) | giant blobs (far right).
# Then set PP_MIN_*/PP_MAX_* in section 3b and run section 11.
if MODE == "full":
    import numpy as np
    import matplotlib.pyplot as plt
    import pandas as pd
    stitch_dir = output_dir / "stage4_full_stitch"
    prov = stitch_dir / "label_provenance.csv"
    if prov.exists():
        vc = pd.read_csv(prov)["voxel_count"].to_numpy()
        vc = vc[vc > 0]
        vox_um3 = float(np.prod(PP_VOXEL_SIZE_UM)) if PP_VOXEL_SIZE_UM else None
        print(f"{len(vc):,} cells | voxel_count percentiles"
              + (f"  (voxel = {vox_um3:.4g} um^3)" if vox_um3 else "") + ":")
        for p in (1, 5, 50, 95, 99, 99.9):
            v = int(np.percentile(vc, p))
            print(f"   p{p:>5}: {v:>11,d} vox" + (f"   (~{v*vox_um3:.3g} um^3)" if vox_um3 else ""))
        # resolve the cuts you'd apply (percentile overrides absolute)
        eff_min = int(np.percentile(vc, PP_MIN_PERCENTILE)) if PP_MIN_PERCENTILE else PP_MIN_VOXELS
        eff_max = int(np.percentile(vc, PP_MAX_PERCENTILE)) if PP_MAX_PERCENTILE else PP_MAX_VOXELS
        # log-scale histogram (sizes span orders of magnitude)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.hist(np.log10(vc), bins=80, color="#4477aa", alpha=0.85)
        ax.set_xlabel("log10(voxel_count)")
        ax.set_ylabel("number of cells")
        ax.set_title("Cell-size distribution (raw stitch) - set thresholds from the gaps")
        def _vline(v, label, color):
            if v and v > 0:
                ax.axvline(np.log10(max(v, 1)), color=color, ls="--", lw=1.4, label=f"{label}: {v:,}")
        if PP_REMOVE_SMALL: _vline(eff_min, "small cut", "#cc6677")
        if PP_REMOVE_LARGE: _vline(eff_max, "large cut", "#882255")
        _vline(int(np.percentile(vc, 50)), "median", "#aaaaaa")
        ax.legend(fontsize=8)
        plt.tight_layout(); plt.show()
        n_below = int((vc < eff_min).sum()) if PP_REMOVE_SMALL else 0
        n_above = int((vc > eff_max).sum()) if (PP_REMOVE_LARGE and eff_max > 0) else 0
        print(f"\nWith current section-3b settings: small cut would drop {n_below:,} | "
              f"large cut would drop {n_above:,}")
    else:
        print("No label_provenance.csv yet - run the stitch (section 9) first.")
else:
    print("Size distribution is a Mode-B ('full') feature.")

82,759 cells | voxel_count percentiles:
   p    1:          44 vox
   p    5:         815 vox
   p   50:       6,953 vox
   p   95:      15,695 vox
   p   99:      20,263 vox
   p 99.9:      29,584 vox


<Figure size 900x400 with 1 Axes>


With current section-3b settings: small cut would drop 1,749 | large cut would drop 0


## 11. Post-Stitch Cleanup + Smooth  *(full mode only)*

Optional QC pass on the stitched volume: drop tiny-fragment confetti (size filter) + detached debris (island removal), and apply the Step-3 topology-preserving SDF smooth per instance. Writes `global_labels_clean.tif/.npy` + `label_provenance_clean.csv` next to the stitch (non-destructive). Set `PP_DRY_RUN = True` first to calibrate `PP_MIN_VOXELS` on the real voxel-count distribution before the full write.

In [ ]:
# ===== Run post-stitch cleanup + smooth =====
# Tweak in isolation: set PP_DRY_RUN=True (section 3b), run THIS cell, read the logged
# distribution + drop counts (or use section 10b's plot), set the thresholds, then
# re-run with PP_DRY_RUN=False. Idempotent: delete global_labels_clean.tif to force a rerun.
if MODE == "full" and RUN_POSTPROCESS:
    from packages.epicycle_scaledseg.core.postprocess_stage import PostprocessStage
    stitch_dir = output_dir / "stage4_full_stitch"
    pp_stage = PostprocessStage(
        input_dir=str(stitch_dir), output_dir=str(stitch_dir),
        remove_small=PP_REMOVE_SMALL, min_voxels=PP_MIN_VOXELS, min_percentile=PP_MIN_PERCENTILE,
        remove_large=PP_REMOVE_LARGE, max_voxels=PP_MAX_VOXELS, max_percentile=PP_MAX_PERCENTILE,
        remove_islands=PP_REMOVE_ISLANDS, island_downsample=PP_ISLAND_DOWNSAMPLE,
        island_min_component_frac=PP_ISLAND_MIN_FRAC, island_dilate=PP_ISLAND_DILATE,
        smooth=PP_SMOOTH, smooth_sigma=PP_SMOOTH_SIGMA,
        smooth_sigma_z=PP_SMOOTH_SIGMA_Z, smooth_sigma_xy=PP_SMOOTH_SIGMA_XY,
        preserve_topology=PP_PRESERVE_TOPOLOGY, n_workers=PP_WORKERS,
        voxel_size_um=PP_VOXEL_SIZE_UM, dry_run=PP_DRY_RUN,
    )
    pp_res = pp_stage.run([stitch_dir], n_gpus=n_gpus)
    print(pp_res.summary())
    man = stitch_dir / "postprocess_manifest.json"
    if man.exists():
        import json
        m = json.loads(man.read_text())
        print(f"  {m['n_cells_before']} -> {m['n_cells_after']} cells "
              f"(dropped {m['n_dropped_size']} small + {m.get('n_dropped_large', 0)} large + "
              f"{m['n_dropped_island']} island); smooth={m['smooth']}  dry_run={m['dry_run']}")
        print(f"  effective cuts: <{m.get('eff_min_voxels')} (small) / >{m.get('eff_max_voxels')} (large) voxels")
    if not PP_DRY_RUN:
        clean = stitch_dir / "global_labels_clean.tif"
        print(f"  clean output -> {clean}  ({'exists' if clean.exists() else 'MISSING'})")
else:
    print("Postprocess skipped (needs MODE='full' + RUN_POSTPROCESS=True).")

2026-06-19 15:13:23,245 epicycle.postprocess INFO postprocess: global_labels_clean.tif exists + manifest complete — skipping (idempotent).


1/1 succeeded (0.0s)
  87411 -> 80169 cells (dropped 6401 small + 0 large + 841 island); smooth=True  dry_run=False
  effective cuts: <200 (small) / >0 (large) voxels
  clean output -> <run>/epicycle_output_nuclei/stage4_full_stitch/global_labels_clean.tif  (exists)


## 12. Per-Cell Feature Extraction  *(full mode only — opt-in)*

Opt-in per-cell QC / morphometry on the cleaned whole-tissue volume (the analogue of EpiCycle's Step-3 QC features; column names mirror the plugin's `qc_features.py`). Writes `cell_features.parquet` + `.csv` + `feature_manifest.json` next to the stitch. Per-cell bbox-parallel (never loads the full volume); idempotent. Size the job first with `FEAT_DRY_RUN = True` (+ `FEAT_INCLUDE_SURFACE = False`) in section 3c. See `docs/feature_extraction_guide.md`.

In [ ]:
# ===== Run per-cell feature extraction =====
# Opt-in + after postprocess. Size the job first: FEAT_DRY_RUN=True (+ FEAT_INCLUDE_SURFACE=False)
# in section 3c, run THIS cell, read the cell count, then re-run with FEAT_DRY_RUN=False.
# Idempotent: delete cell_features.parquet to force a rerun.
if MODE == "full" and RUN_FEATURES:
    from packages.epicycle_scaledseg.core.feature_stage import FeatureStage
    stitch_dir = output_dir / "stage4_full_stitch"
    feat_stage = FeatureStage(
        input_dir=str(stitch_dir), output_dir=str(stitch_dir),
        prefer_clean=FEAT_PREFER_CLEAN, voxel_size_um=FEAT_VOXEL_SIZE_UM,
        min_voxels=FEAT_MIN_VOXELS, include_surface=FEAT_INCLUDE_SURFACE,
        invert_z=FEAT_INVERT_Z, k_neighbors=FEAT_K_NEIGHBORS,
        density_radius=FEAT_DENSITY_RADIUS, n_workers=FEAT_WORKERS,
        dry_run=FEAT_DRY_RUN,
    )
    feat_res = feat_stage.run([stitch_dir], n_gpus=n_gpus)
    print(feat_res.summary())
    man = stitch_dir / "feature_manifest.json"
    if man.exists():
        import json
        m = json.loads(man.read_text())
        print(f"  status={m['status']}  requested={m.get('n_cells_requested')}  "
              f"extracted={m.get('n_cells_extracted')}  missing={m.get('n_missing')}  "
              f"failed={m.get('n_failed')}  clean_inputs={m.get('used_clean_inputs')}")
        fams = m.get('families', {})
        if fams:
            print('  families: ' + ', '.join(f'{k}({len(v)})' for k, v in fams.items()))
    if not FEAT_DRY_RUN:
        tbl = stitch_dir / "cell_features.parquet"
        print(f"  feature table -> {tbl}  ({'exists' if tbl.exists() else 'MISSING'})")
else:
    print("Features skipped (needs MODE='full' + RUN_FEATURES=True; runs after postprocess).")


2026-06-19 15:13:23,649 epicycle.features INFO features: 80169 provenance rows -> 80169 cells to extract (min_voxels=1, surface=True, clean_inputs=True); voxel_count {'n': 80169, 'min': 2, 'p1': 246, 'p5': 1196, 'median': 6716, 'p95': 15292, 'p99': 19798, 'max': 59565}; 1 voxel = 0.01758 um^3
2026-06-19 15:13:23,651 epicycle.features INFO features: extracting 80169 cell(s) on 23 worker(s)
features extract: 100%|████████████████████████████████████████████| 80169/80169 [00:56<00:00, 1408.63it/s]
2026-06-19 15:14:24,578 epicycle.features INFO features complete: 80169 cells -> cell_features.parquet + cell_features.csv (0 missing, 0 failed)


1/1 succeeded (0.0s)
  status=complete  requested=80169  extracted=80169  missing=0  failed=0  clean_inputs=True
  families: morphology(18), surface(4), discrete_compactness(1), skeleton(3), cross_section(6), crowding(4), provenance(5)
  feature table -> <run>/epicycle_output_nuclei/stage4_full_stitch/cell_features.parquet  (exists)


In [ ]:
# Quick status check — how many tiles completed each stage?
import json

# Re-discover the real tile dirs (tiling replaces the placeholder during the run).
if tiles_dir.is_dir():
    tile_dirs = sorted([
        str(p) for p in tiles_dir.iterdir()
        if p.is_dir() and not p.name.startswith(".")
    ])

seg_root = output_dir / "stage1_segmentation"
recon_root = output_dir / "stage2_reconstruction"

seg_done = sum(1 for td in tile_dirs if (seg_root / Path(td).name / "SEGMENTATION_DONE").exists())
recon_done = sum(1 for td in tile_dirs if (recon_root / Path(td).name / f"{Path(td).name}_mask.tif").exists())

print(f"Mode: {MODE}")
print(f"Stage 1 (Segmentation):   {seg_done}/{len(tile_dirs)} complete")
print(f"Stage 2 (Reconstruction): {recon_done}/{len(tile_dirs)} complete")

if MODE == "coarse":
    print(f"Stage 3 (Assembly):       check {output_dir / 'stage3_assembly'}")
    g = output_dir / 'stage4_global' / 'global_segmentation.tif'
    print(f"Stage 4 (Global Recon):   {'done' if g.exists() else 'not done'}")
else:  # full
    stitch_dir = output_dir / 'stage4_full_stitch'
    labels = stitch_dir / 'global_labels.tif'
    manifest = stitch_dir / 'result_manifest.json'
    print(f"Mode B (IoU stitch):      {'done' if labels.exists() else 'not done'} -> {stitch_dir}")
    if manifest.exists():
        m = json.loads(manifest.read_text())
        print(f"    status={m.get('status')}  n_global_labels={m.get('n_global_labels')}  "
              f"dtype={m.get('dtype')}  outputs: global_labels.tif + .npy")
        # Seam QC headline (present when EMIT_PROVENANCE was on).
        ss = m.get('seam_summary')
        if ss:
            print(f"    seam QC:  clean={ss.get('clean')}  seam={ss.get('seam')}  "
                  f"corner={ss.get('corner')}  border_clipped={ss.get('border_clipped')}")
            print(f"    worklist: {m.get('label_provenance')}")

Mode: full
Stage 1 (Segmentation):   42/42 complete
Stage 2 (Reconstruction): 42/42 complete
Mode B (IoU stitch):      done -> <run>/epicycle_output_nuclei/stage4_full_stitch
    status=complete  n_global_labels=87411  dtype=int32  outputs: global_labels.tif + .npy
    seam QC:  clean=41744  seam=37034  corner=8633  border_clipped=1192
    worklist: <run>/epicycle_output_nuclei/stage4_full_stitch/label_provenance.csv


In [ ]:
# ===== Inspect the result (RAM-bounded — memmap + one slice, no full-volume load) =====
import numpy as np

if MODE == "full":
    stitch_dir = output_dir / "stage4_full_stitch"
    npy = stitch_dir / "global_labels.npy"
    if npy.exists():
        vol = np.load(npy, mmap_mode="r")   # memmap: does NOT load the full volume
        print(f"global_labels: shape={vol.shape}, dtype={vol.dtype}")
        mid = np.asarray(vol[vol.shape[0] // 2])          # one Z slice into RAM
        n_mid = int(len(np.unique(mid)) - 1)              # minus background 0
        print(f"  middle slice: {n_mid} distinct labels, max id = {int(mid.max())}")
        print(f"  total instances: see result_manifest n_global_labels (status cell above)")
    else:
        print(f"No global_labels.npy at {npy} - run with MODE='full' first.")

    # --- QC worklist preview: the seam/corner cells most likely mis-stitched ---
    # label_provenance.csv is the per-cell QC table. The seam/corner cells are the
    # suspects; min_match_iou ranks them (low = the stitch was least confident this
    # merge was right, so review first). The ZYX bbox locates a cell for a review
    # crop in napari later. voxel_count==0 = a cell merged/clipped out of the volume.
    prov = stitch_dir / "label_provenance.csv"
    if prov.exists():
        import pandas as pd
        df = pd.read_csv(prov)
        suspects = df[df["seam_status"].isin(["seam", "corner"]) & (df["voxel_count"] > 0)]
        n_clean = int((df["seam_status"] == "clean").sum())
        print(f"\nlabel_provenance.csv: {len(df)} cells "
              f"({n_clean} clean, {len(suspects)} seam/corner suspects)")
        if len(suspects):
            cols = ["dense_id", "seam_status", "min_match_iou", "voxel_count",
                    "z_min", "z_max", "y_min", "y_max", "x_min", "x_max", "touches_border"]
            worst = suspects.sort_values("min_match_iou").head(15)[cols]
            print("  lowest-confidence seam/corner cells (review these first):")
            print(worst.to_string(index=False))
    else:
        print("\n(no label_provenance.csv - EMIT_PROVENANCE was off this run)")
else:
    g = output_dir / "stage4_global" / "global_segmentation.tif"
    if g.exists():
        import tifffile
        with tifffile.TiffFile(str(g)) as tf:
            s = tf.series[0]
            print(f"global_segmentation (downsampled): shape={s.shape}, dtype={s.dtype}")
    else:
        print(f"No global_segmentation.tif at {g} - run with MODE='coarse' first.")

global_labels: shape=(266, 5631, 4709), dtype=int32
  middle slice: 12666 distinct labels, max id = 86926
  total instances: see result_manifest n_global_labels (status cell above)

label_provenance.csv: 87411 cells (41744 clean, 45270 seam/corner suspects)
  lowest-confidence seam/corner cells (review these first):
 dense_id seam_status  min_match_iou  voxel_count  z_min  z_max  y_min  y_max  x_min  x_max  touches_border
    40178      corner       0.250000        11736     30     54   3322   3354   2352   2385               0
    12817      corner       0.250000           46      0      6   1765   1772   2357   2361               1
    21178        seam       0.250000           86      2      6   1880   1887   1634   1639               0
    50081        seam       0.250317         6385     47     70   3362   3386   1516   1546               0
      390        seam       0.250346          181     65     73    955    961    709    714               0
    12351        seam       0.2504